In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/datasets/nhatduc2425/pathvqa-rag/pathvqa_fusion.index
/kaggle/input/datasets/nhatduc2425/pathvqa-rag/pathvqa_metadata.pt
/kaggle/input/datasets/nhatduc2425/pathvqa-rag/pathvqa_gated_fusion.pth


In [2]:
!pip install faiss-cpu open_clip_torch accelerate qwen-vl-utils evaluate rouge_score bert_score scikit-learn -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 64.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.1/61.1 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 56.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00


In [3]:
import torch
import torch.nn as nn
import faiss
import open_clip
import time
import evaluate
import numpy as np
import gc
import os
import random
from sklearn.metrics import accuracy_score, f1_score
from datasets import load_dataset
from PIL import Image, ImageFile
from transformers import pipeline

# Ép hệ thống bỏ qua lỗi ảnh thiếu byte của PathVQA
ImageFile.LOAD_TRUNCATED_IMAGES = True

# --- 1. Đường dẫn tuyệt đối dựa trên Input Dataset Kaggle ---
SAVE_DIR = "/kaggle/input/datasets/nhatduc2425/pathvqa-rag"

# Cố định đúng Seed 42 để đảm bảo quét trúng 500 mẫu y hệt như lượt chạy trước
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)

# --- 2. Kiến trúc Gated Fusion Core ---
class GatedFusion(nn.Module):
    def __init__(self, embedDim=512):
        super(GatedFusion, self).__init__()
        self.gateLayer = nn.Linear(embedDim * 2, embedDim)
        self.sigmoidFunc = nn.Sigmoid()
        self.fusionLayer = nn.Linear(embedDim * 2, embedDim)

    def forward(self, imageEmbeds, textEmbeds):
        combinedEmbeds = torch.cat((imageEmbeds, textEmbeds), dim=1)
        gateValue = self.sigmoidFunc(self.gateLayer(combinedEmbeds))
        gatedImage = imageEmbeds * gateValue
        gatedText = textEmbeds * (1 - gateValue)
        fusedOutput = self.fusionLayer(torch.cat((gatedImage, gatedText), dim=1))
        fusedOutput = fusedOutput / fusedOutput.norm(dim=-1, keepdim=True)
        return fusedOutput

# --- 3. Lớp RAG Retriever tối ưu trên Kaggle ---
class PathVqaRetriever:
    def __init__(self, saveDirectory=SAVE_DIR):
        self.device = "cuda:0" if torch.cuda.is_available() else "cpu"
        self.faissIndex = faiss.read_index(f"{saveDirectory}/pathvqa_fusion.index")
        self.metaDataList = torch.load(f"{saveDirectory}/pathvqa_metadata.pt")
        
        modelName = 'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
        self.bioMedClip, _, self.preprocessImage = open_clip.create_model_and_transforms(modelName)
        self.tokenizer = open_clip.get_tokenizer(modelName)
        self.bioMedClip.to(self.device).eval()
        
        self.fusionModel = GatedFusion(embedDim=512).to(self.device)
        self.fusionModel.load_state_dict(torch.load(f"{saveDirectory}/pathvqa_gated_fusion.pth"))
        self.fusionModel.eval()

    def retrieveTopK(self, imageObj, questionText, topK=4): # Đã sửa k=4
        with torch.no_grad():
            imgRGB = imageObj.convert("RGB") if imageObj.mode != "RGB" else imageObj
            procImg = self.preprocessImage(imgRGB).unsqueeze(0).to(self.device)
            tokens = self.tokenizer([questionText]).to(self.device)
            
            imgFeat = self.bioMedClip.encode_image(procImg)
            txtFeat = self.bioMedClip.encode_text(tokens)
            imgFeat /= imgFeat.norm(dim=-1, keepdim=True)
            txtFeat /= txtFeat.norm(dim=-1, keepdim=True)

            queryVec = self.fusionModel(imgFeat, txtFeat).cpu().numpy().astype('float32')
            distances, indices = self.faissIndex.search(queryVec, topK)
            return [self.metaDataList[idx] for idx in indices[0]], distances[0]

# --- 4. Lớp SLM Generator gánh vác Qwen2-VL-2B-Instruct song song ---
class SlmGenerator:
    def __init__(self):
        print("\n[Hạ tầng] Đang tải Qwen2-VL-2B-Instruct và phân bổ trên cấu hình Kaggle...")
        # Sử dụng device_map="auto" trên môi trường T4x2 để đối chứng công bằng với 7B
        self.pipe = pipeline(
            "image-text-to-text",
            model="Qwen/Qwen2-VL-2B-Instruct",
            device_map="auto",
            model_kwargs={"torch_dtype": torch.bfloat16} 
        )
        print("Đã nạp thành công mô hình cấu hình 2B!")

    def optimizeImage(self, pilImg, maxRes=336):
        if pilImg.mode != "RGB": 
            pilImg = pilImg.convert("RGB")
        pilImg.thumbnail((maxRes, maxRes), Image.Resampling.LANCZOS)
        return pilImg

    def generate(self, testImg, testQuest, contexts, trainDS):
        contentBlock = []
        
        systemInstruction = (
            "<System>\n"
            "You are an expert medical diagnostic assistant. Your task is to answer clinical questions "
            "based on an input pathology image and a set of retrieved similar cases.\n"
            "Strictly follow these rules:\n"
            "1. Analyze the provided [Image] and [Question].\n"
            "2. Evaluate the [Retrieved Evidence] from the knowledge base.\n"
            "Note that retrieved cases might have conflicting answers; use them as reference, "
            "but prioritize the visual features of the current image.\n"
            "3. Provide a concise and definitive answer.\n"
            "</System>\n\n"
            "<User>\n"
            "[Retrieved Evidence (Top-K)]\n"
            "Below are similar past cases from the clinical database:\n"
        )
        contentBlock.append({"type": "text", "text": systemInstruction})
        
        for i, ctx in enumerate(contexts):
            try:
                optimizedTrainImg = self.optimizeImage(trainDS[ctx['id']]['image'])
                contentBlock.append({"type": "image", "image": optimizedTrainImg})
                evidenceText = f"{i+1}. Case Reference: Question \"{ctx['question']}\" resulted in Answer \"{ctx['answer']}\".\n"
                contentBlock.append({"type": "text", "text": evidenceText})
            except Exception:
                continue
            
        contentBlock.append({"type": "text", "text": "\n[Target Case Image]\n"})
        optimizedTestImg = self.optimizeImage(testImg)
        contentBlock.append({"type": "image", "image": optimizedTestImg})
        
        targetInstruction = (
            f"\nCurrent Question: {testQuest}\n\n"
            "Based on the visual evidence and the clinical context provided above, what is the correct answer for the current case?\n"
            "</User>\n\n"
            "<Assistant>\n"
            "Final Answer: "
        )
        contentBlock.append({"type": "text", "text": targetInstruction})
        
        messages = [{"role": "user", "content": contentBlock}]
        outputs = self.pipe(
            messages, 
            max_new_tokens=15, 
            generate_kwargs={"do_sample": False, "max_length": None}
        )
        
        rawResponse = outputs[0]['generated_text']
        finalText = rawResponse[-1]['content'] if isinstance(rawResponse, list) else rawResponse
        return finalText.replace("</Assistant>", "").strip()

# --- 5. Thực thi Pipeline Đánh giá tái lập ---
def runKagglePathVqa2bFairBenchmark(sampleSize=500):
    print("1. Khởi tạo các thư viện đo lường (Metrics)...")
    bleuMetric = evaluate.load("bleu")
    rougeMetric = evaluate.load("rouge")
    bertscoreMetric = evaluate.load("bertscore")
    
    print("2. Tải các mô hình và kho dữ liệu PathVQA...")
    retriever = PathVqaRetriever()
    slm = SlmGenerator()
    testDS = load_dataset("flaviagiammarino/path-vqa", split="test")
    trainDS = load_dataset("flaviagiammarino/path-vqa", split="train")
    
    totalIndices = list(range(len(testDS)))
    sampledIndices = random.sample(totalIndices, min(sampleSize, len(testDS)))
    
    results = {"closed": {"ref": [], "pred": []}, "open": {"ref": [], "pred": []}}
    latencyList = []
    
    yesKeywords = ['yes', 'correct', 'true', 'positive', 'present', 'visible']
    noKeywords = ['no', 'incorrect', 'false', 'negative', 'absent', 'normal', 'none']
    
    print(f"\n🚀 BẮT ĐẦU CHẠY THỰC NGHIỆM ĐỐI CHỨNG CÔNG BẰNG (QWEN-2B) TRÊN {len(sampledIndices)} MẪU KAGGLE VỚI K=4...")
    print("-" * 60)
    
    if torch.cuda.is_available(): 
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()
        
    startTotalTime = time.time()

    for progressCount, idx in enumerate(sampledIndices):
        item = testDS[idx]
        startTimeSample = time.time()
        
        try:
            # Đã sửa k=4 tại hàm gọi này
            contexts, _ = retriever.retrieveTopK(item['image'], item['question'], topK=4)
            aiAnswerRaw = slm.generate(item['image'], item['question'], contexts, trainDS)
        except Exception:
            continue
            
        latencyList.append(time.time() - startTimeSample)
        
        aiAnswer = aiAnswerRaw.lower().strip()
        groundTruth = str(item['answer']).lower().strip()
        
        if groundTruth in ['yes', 'no']:
            results["closed"]["ref"].append(groundTruth)
            hasYes = any(word in aiAnswer for word in yesKeywords)
            hasNo = any(word in aiAnswer for word in noKeywords)
            
            if hasYes and not hasNo:
                predClean = 'yes'
            elif hasNo and not hasYes:
                predClean = 'no'
            else:
                firstYesPos = aiAnswer.find('yes') if 'yes' in aiAnswer else 9999
                firstNoPos = aiAnswer.find('no') if 'no' in aiAnswer else 9999
                predClean = 'yes' if firstYesPos < firstNoPos else ('no' if firstNoPos < firstYesPos else 'unknown')
            results["closed"]["pred"].append(predClean)
        else:
            results["open"]["ref"].append(groundTruth)
            results["open"]["pred"].append(aiAnswer)
            
        if (progressCount + 1) % 50 == 0: 
            print(f"Tiến độ Kaggle (2B): {progressCount + 1}/{len(sampledIndices)} mẫu...")

    endTotalTime = time.time()
    
    # --- XUẤT BÁO CÁO THỰC NGHIỆM CHÍNH THỨC PHIÊN BẢN 2B TRÊN KAGGLE ---
    print("\n" + "="*60 + "\n📊 BÁO CÁO THỰC NGHIỆM ĐỐI CHỨNG CÔNG BẰNG - LIGHT-MVQA (QWEN-2B, K=4)\n" + "="*60)
    
    if results["closed"]["ref"]:
        accScore = accuracy_score(results["closed"]["ref"], results["closed"]["pred"])
        f1Weighted = f1_score(results["closed"]["ref"], results["closed"]["pred"], average='weighted', zero_division=0)
        print(f"[1. Closed-Ended Classification (Total: {len(results['closed']['ref'])} samples)]")
        print(f"  - Accuracy : {accScore:.4f}")
        print(f"  - F1-Score : {f1Weighted:.4f}")
    
    if results["open"]["ref"]:
        print(f"\n[2. Open-Ended Text Generation (Total: {len(results['open']['ref'])} samples)]")
        predsList = results["open"]["pred"]
        refsList = [[r] for r in results["open"]["ref"]]
        
        bleu1 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=1, smooth=True)['bleu']
        bleu2 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=2, smooth=True)['bleu']
        bleu3 = bleuMetric.compute(predictions=predsList, references=refsList, max_order=3, smooth=True)['bleu']
        rougeL = rougeMetric.compute(predictions=predsList, references=results["open"]["ref"])['rougeL']
        
        bertResults = bertscoreMetric.compute(predictions=predsList, references=results["open"]["ref"], lang="en")
        print(f"  - BLEU-1   : {bleu1:.4f}")
        print(f"  - BLEU-2   : {bleu2:.4f}")
        print(f"  - BLEU-3   : {bleu3:.4f}")
        print(f"  - ROUGE-L  : {rougeL:.4f}")
        print(f"  - BERTScore: {np.mean(bertResults['f1']):.4f}")

    totalDuration = endTotalTime - startTotalTime
    print("\n[3. Computational Efficiency Measurements]")
    print(f"  - Inference Latency (Độ trễ trung bình) : {np.mean(latencyList):.2f} s/mẫu")
    print(f"  - Throughput (Tốc độ xử lý)            : {len(latencyList) / totalDuration:.2f} mẫu/giây")
    
    if torch.cuda.is_available():
        print("\n[4. Multi-GPU Memory Profile]")
        for gpuId in range(torch.cuda.device_count()):
            vramMB = torch.cuda.max_memory_allocated(device=gpuId) / (1024 ** 2)
            print(f"  - Peak Memory Usage on GPU {gpuId}      : {vramMB:.2f} MB")
    print("="*60)
    
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()

if __name__ == "__main__":
    runKagglePathVqa2bFairBenchmark(sampleSize=500)

1. Khởi tạo các thư viện đo lường (Metrics)...


2. Tải các mô hình và kho dữ liệu PathVQA...


open_clip_config.json:   0%|          | 0.00/707 [00:00<?, ?B/s]

open_clip_pytorch_model.bin:   0%|          | 0.00/784M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/385 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/28.0 [00:00<?, ?B/s]

... (43 outputs left out)

model.safetensors:   0%|          | 0.00/1.42G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/389 [00:00<?, ?it/s]

RobertaModel LOAD REPORT from: roberta-large
Key                             | Status     | 
--------------------------------+------------+-
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.bias                    | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
pooler.dense.bias               | MISSING    | 
pooler.dense.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


  - BLEU-1   : 0.0713
  - BLEU-2   : 0.0313
  - BLEU-3   : 0.0184
  - ROUGE-L  : 0.2161
  - BERTScore: 0.8693

[3. Computational Efficiency Measurements]
  - Inference Latency (Độ trễ trung bình) : 2.72 s/mẫu
  - Throughput (Tốc độ xử lý)            : 0.37 mẫu/giây

[4. Multi-GPU Memory Profile]
  - Peak Memory Usage on GPU 0      : 4003.03 MB
  - Peak Memory Usage on GPU 1      : 2230.60 MB
